In [4]:
import random
import os, sys, pandas as pd
import numpy as np
username = os.getlogin()
sys.path.append(rf'/home/{username}/data_share')
# from functions import *

import FactorFramework.FactorFramework as ff
from tqdm import tqdm

ff.set_gpu_device('cuda:0')
from sklearn.preprocessing import StandardScaler

In [2]:
# 采样获得时间和日期样本
trade_day_list = os.listdir("/home/user61/data_share/Stock60sBaseDataAll/Feather/Base/")
trade_day_list = [i.split('.')[0] for i in trade_day_list]

second_list = pd.read_feather("/home/user61/data_share/Stock60sBaseDataAll/Feather/Base/20210201.fea")["second"].unique()
second_list = list(second_list)

dir_list = os.listdir('/home/user61/data_share/Stock60sBaseDataAll/Feather/')

In [26]:
def get_cross_section_data_(target_dir: str, target_date: str, target_time: int) -> pd.DataFrame:
    """获取单个文件夹里面的结构数据

    Args:
        target_dir (str): 目标因子文件夹
        target_data (str): 采样日期
        target_time (int): 采样时间点

    Returns:
        pd.DataFrame: _description_
    """
    feather_path = f"/home/user61/data_share/Stock60sBaseDataAll/Feather/{target_dir}/{target_date}.fea"
    feather_data = pd.read_feather(feather_path)
    
    # 选取目标时间点的数据
    feather_data = feather_data.loc[feather_data['second'] == target_time, :]
    feather_data = feather_data.drop(columns=['second']).reset_index(drop=True)
    
    return feather_data


def get_cross_section_data(target_dirs: list, target_date: str, target_time: int) -> pd.DataFrame:
    
    tol_feather_data = None
    for target_dir in target_dirs:
        # 跳过Test文件夹
        if 'test' in target_dir.lower():
            continue
        # 遍历文件夹
        try:
            feather_data = get_cross_section_data_(target_dir, target_date, target_time)
            if tol_feather_data is None:
                tol_feather_data = feather_data
            else:
                conflict_cols = set(feather_data.columns) & set(tol_feather_data.columns) - set(['code'])
                tol_feather_data = pd.merge(tol_feather_data, feather_data.drop(columns=conflict_cols), how='outer', on=['code'])
                
            # print(target_dir, 'Close' in tol_feather_data.columns)
        except Exception as e:
            # print(f"Error processing {target_dir}: {e}")
            continue
    
    return tol_feather_data.reset_index(drop=True)


def cal_correlation_matrix(target_dirs: list, target_date: str, target_time: int, method: str='pearson') -> pd.DataFrame:
    """计算相关性矩阵

    Args:
        target_dirs (list): 目标因子文件夹
        target_date (str): 采样日期
        target_time (int): 采样时间点

    Returns:
        pd.DataFrame: _description_
    """
    tol_feather_data = get_cross_section_data(target_dirs, target_date, target_time)
    tol_feather_data = tol_feather_data.drop(columns=["code"])
    
    # print(tol_feather_data.head())
    
    # 使用zscore方法进行标准化放缩
    tol_feather_data.loc[:, :] = np.where(np.isinf(tol_feather_data.loc[:, :].values), np.nan, tol_feather_data.loc[:, :].values)
    tol_feather_data = tol_feather_data.fillna(tol_feather_data.median(numeric_only=True))
    
    stds = tol_feather_data.std(axis=0)
    tol_feather_data = tol_feather_data.loc[:, stds != 0]  # 防止除以0错误
    
    std_scaler = StandardScaler()
    tol_feather_data.loc[:, :] = std_scaler.fit_transform(tol_feather_data)
    
    # 计算相关性矩阵
    tol_feather_data = tol_feather_data.corr(method=method)
    
    return tol_feather_data


def cal_field_mean(target_dirs: list, target_date: str, target_time: int) -> pd.DataFrame:
    """计算字段均值

    Args:
        target_dirs (list): _description_
        target_date (str): _description_
        target_time (int): _description_
        save_path (str): _description_

    Returns:
        pd.DataFrame: _description_
    """
    tol_feather_data = get_cross_section_data(target_dirs, target_date, target_time)
    
    # 计算均值
    mean_data = tol_feather_data.drop(columns=['code']).mean().to_frame().T
    
    return mean_data


def cal_field_stddev(target_dirs: list, target_date: str, target_time: int) -> pd.DataFrame:
    """计算字段均值

    Args:
        target_dirs (list): _description_
        target_date (str): _description_
        target_time (int): _description_
        save_path (str): _description_

    Returns:
        pd.DataFrame: _description_
    """
    tol_feather_data = get_cross_section_data(target_dirs, target_date, target_time)
    
    # 计算均值
    stddev_data = tol_feather_data.drop(columns=['code']).std().to_frame().T
    
    return stddev_data

In [27]:
# get_cross_section_data(dir_list, '20240909',143000000)

In [28]:
res = cal_correlation_matrix(dir_list, '20240909',143000000)
# res.reset_index().to_feather('/home/user61/MyWorkSpace/08_gp_v2/DEAP/correlation_matrix_119_demo.fea')
res.head()

,TradeTotalPower_100ms_B,TradeDomFreq_100ms_10_B,TradeDomFreq_100ms_50_B,TradeDomFreq_100ms_100_B,TradeTotalPower_100ms_S,TradeDomFreq_100ms_10_S,TradeDomFreq_100ms_50_S,TradeDomFreq_100ms_100_S,CancelNum_5s_B,CancelNum_1min_B,...,OrderVol_first_digit_8_Num_S,OrderVol_first_digit_9_Num_S,trade_proportion_buy_mean,trade_proportion_sell_mean,OrderAmtMin_B,OrderAmtMin_S,TransWaitMean_B,TransWaitMean_S,reach_up_limit,reach_down_limit
TradeTotalPower_100ms_B,1.000000,0.961385,0.978944,0.987201,0.167553,0.184408,0.176098,0.174297,0.121521,0.230697,...,0.213264,0.281103,0.089819,0.074235,-0.046007,-0.034740,-0.010512,0.063135,0.052621,0.018218
TradeDomFreq_100ms_10_B,0.961385,1.000000,0.996792,0.991901,0.189650,0.207736,0.198636,0.197046,0.131441,0.247057,...,0.234796,0.313723,0.102884,0.079674,-0.043822,-0.032756,-0.015476,0.067147,0.075851,0.011818
TradeDomFreq_100ms_50_B,0.978944,0.996792,1.000000,0.998827,0.184509,0.202277,0.193394,0.191740,0.129168,0.243756,...,0.229103,0.306341,0.098793,0.077872,-0.044996,-0.033517,-0.014477,0.066445,0.069345,0.013964
TradeDomFreq_100ms_100_B,0.987201,0.991901,0.998827,1.000000,0.181127,0.198888,0.190056,0.188327,0.128056,0.242283,...,0.226339,0.302260,0.096907,0.077359,-0.045595,-0.033961,-0.013799,0.065798,0.065155,0.015305
TradeTotalPower_100ms_S,0.167553,0.189650,0.184509,0.181127,1.000000,0.971446,0.981696,0.988031,0.036891,0.069175,...,0.231398,0.286756,0.055395,0.080310,-0.040703,-0.032818,0.086744,-0.009912,0.105209,-0.009370


In [29]:
# 做采样
import itertools
import random
random.seed(100)

day_second_comb = list(itertools.product(
    random.sample(trade_day_list, 256), random.sample(second_list, 4)
))

joint_col = set()
corr_data = list()

for day, second in tqdm(day_second_comb, desc="抽样计算进度"):
    # 遍历采样对象
    res = cal_correlation_matrix(dir_list, day, second)
    corr_data.append(res)
    joint_col.update(res.columns)
    
joint_col = list(joint_col)
array_lst = list()

for this_data in corr_data:
    # 变量相关系数矩阵
    array_lst.append(
        this_data.reindex(index=joint_col, columns=joint_col).values
    )
    
mean_corr = np.nanmean(np.stack(array_lst, axis=-1), axis=2)
mean_corr_df = pd.DataFrame(data=mean_corr, index=joint_col, columns=joint_col)

抽样计算进度: 100%|██████████| 1024/1024 [17:10<00:00,  1.01s/it]


In [30]:
mean_corr_df

,OrderAmtsma_B,OrderVol_first_digit_8_Num_S,TradeVolMax_S,cd_iu_sv,TradeDomFreq_100ms_50_B,TradeDomFreq_100ms_10_B,OrderPriceStd_B,OrderAmtMin_B,sp1,CancelNum_10min_B,...,OrderVol_first_digit_2_Num_B,OrderVol_first_digit_7_Num_B,cd_ou_sv,OrderVolbig_S,OrderVol_first_digit_6_Num_S,OrderVol_first_digit_9_Num_B,OrderAmtbig_S,CancelVolMax_B,TradeVolMax_B,volume_eq_500_B
OrderAmtsma_B,1.000000,0.376057,0.139068,0.109845,0.268198,0.287463,0.301959,0.148762,0.431616,0.607646,...,0.718622,0.512708,0.228449,0.232745,0.405195,0.477013,0.532920,0.113912,0.197886,0.630438
OrderVol_first_digit_8_Num_S,0.376057,1.000000,0.291690,0.244469,0.310845,0.325929,0.022144,-0.046406,-0.022417,0.358685,...,0.473137,0.430078,0.314388,0.409943,0.657757,0.418187,0.445601,0.187419,0.248823,0.516391
TradeVolMax_S,0.139068,0.291690,1.000000,0.290329,0.288421,0.290139,-0.030872,-0.064784,-0.087716,0.165681,...,0.225863,0.238265,0.190549,0.572662,0.291065,0.246125,0.335894,0.286617,0.292418,0.226056
cd_iu_sv,0.109845,0.244469,0.290329,1.000000,0.267252,0.271348,-0.022288,-0.040418,-0.056162,0.132655,...,0.182587,0.192885,0.219251,0.457109,0.241534,0.197015,0.228853,0.220042,0.256283,0.173188
TradeDomFreq_100ms_50_B,0.268198,0.310845,0.288421,0.267252,1.000000,0.994419,-0.021675,-0.051458,-0.061329,0.364884,...,0.431765,0.460426,0.208562,0.391009,0.312869,0.472108,0.260998,0.300539,0.783370,0.368049
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
OrderVol_first_digit_9_Num_B,0.477013,0.418187,0.246125,0.197015,0.472108,0.494747,0.006053,-0.054184,-0.033967,0.567230,...,0.644823,0.648668,0.258427,0.346294,0.427004,1.000000,0.360153,0.221128,0.358067,0.655586
OrderAmtbig_S,0.532920,0.445601,0.335894,0.228853,0.260998,0.275762,0.130222,0.025829,0.130355,0.339342,...,0.472174,0.375259,0.281168,0.605592,0.469758,0.360153,1.000000,0.149000,0.233135,0.480817
CancelVolMax_B,0.113912,0.187419,0.286617,0.220042,0.300539,0.302414,-0.019352,-0.049108,-0.074710,0.195425,...,0.189194,0.211214,0.134765,0.292410,0.185146,0.221128,0.149000,1.000000,0.274699,0.170581
TradeVolMax_B,0.197886,0.248823,0.292418,0.256283,0.783370,0.760971,-0.032099,-0.059721,-0.080723,0.268533,...,0.334382,0.351825,0.183608,0.377257,0.249120,0.358067,0.233135,0.274699,1.000000,0.293659


In [11]:
mean_corr_df

,TradeVolMax_S,OrderVol_first_digit_9_Num_S,TradeAmtMax_S,TradePriceStd_S,reach_down_limit,OrderVol_first_digit_8_Num_S,OrderVol_first_digit_7_Num_S,OrderVol_first_digit_4_Num_S,High,bp1,...,OrderAmtMax_S,OrderPriceMean_B,samt15,OrderAmtmid_S,OrderVol_first_digit_7_Num_B,cd_ou_bv,TradeAmtsma_S,OrderVol_first_digit_9_Num_B,TradeDomFreq_100ms_100_S,OrderAmtmid_B
TradeVolMax_S,1.000000,0.296973,0.610944,-0.025378,0.002508,0.291632,0.288443,0.289978,-0.088290,-0.089134,...,0.317623,-0.084560,0.219698,0.210885,0.238203,0.156458,0.196981,0.246067,0.799738,0.178878
OrderVol_first_digit_9_Num_S,0.296973,1.000000,0.308758,0.053687,0.002915,0.637942,0.623809,0.638450,-0.031579,-0.032500,...,0.283279,-0.027032,0.113349,0.531279,0.413316,0.259799,0.425981,0.429369,0.381646,0.411318
TradeAmtMax_S,0.610944,0.308758,1.000000,0.203728,-0.007816,0.321474,0.322267,0.370662,0.180072,0.179449,...,0.535234,0.184309,0.059578,0.470454,0.268564,0.145410,0.510449,0.258513,0.475329,0.411278
TradePriceStd_S,-0.025378,0.053687,0.203728,1.000000,-0.012730,0.066868,0.066112,0.109066,0.579651,0.579066,...,0.178843,0.582701,-0.038208,0.274174,0.070132,0.052859,0.319835,0.058852,-0.012717,0.280614
reach_down_limit,0.002508,0.002915,-0.007816,-0.012730,1.000000,0.002057,0.001938,-0.000057,-0.011224,-0.007836,...,0.018468,-0.010625,0.238728,0.006810,-0.004734,-0.008333,-0.002425,-0.004232,0.009803,-0.003440
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
cd_ou_bv,0.156458,0.259799,0.145410,0.052859,-0.008333,0.276999,0.268902,0.290162,-0.010745,-0.010901,...,0.142655,-0.008934,0.075688,0.295849,0.317647,1.000000,0.207795,0.313459,0.181164,0.355699
TradeAmtsma_S,0.196981,0.425981,0.510449,0.319835,-0.002425,0.446515,0.453623,0.534946,0.374147,0.374269,...,0.489861,0.378600,0.039870,0.762168,0.370665,0.207795,1.000000,0.346330,0.318060,0.662531
OrderVol_first_digit_9_Num_B,0.246067,0.429369,0.258513,0.058852,-0.004232,0.418115,0.416055,0.441859,-0.033918,-0.034816,...,0.260281,-0.029316,0.109408,0.414160,0.648629,0.313459,0.346330,1.000000,0.304277,0.553500
TradeDomFreq_100ms_100_S,0.799738,0.381646,0.475329,-0.012717,0.009803,0.373496,0.369543,0.373022,-0.073154,-0.074074,...,0.370324,-0.069734,0.196440,0.284900,0.299060,0.181164,0.318060,0.304277,1.000000,0.234219


In [31]:
import pickle

with open("/home/user61/MyWorkSpace/08_gp_v2/correlation_matrix_122_demo_128_zscore.pkl", 'wb') as f:
    # 保存相关性矩阵数据
    pickle.dump(mean_corr_df, f)

In [32]:
# 对字段进行抽样
def get_factor_names():
    """
    返回factor_lst = list
    Returns:

    """
    factor_names = list(ff.base_data_config.Base_data_dict.keys())
    factor_names = list(filter(lambda x: x not in ["label4", "alpha4", "label5",
                                                    'reach_up_limit', 'reach_down_limit',
                                                    'valid_mask_label5', 'valid_mask_label4_bar'], factor_names))
    factor_names = random.sample(factor_names, int(np.floor(len(factor_names) * 1.0)))


    return factor_names

In [33]:
len(set(get_factor_names()) - set(joint_col))

159